# Analysis - Perfect Information Scheduler Comparison
Having a closer look at how the Dynamic Scheduler compares to a Static Scheduler that has perfect information (i.e. all of the possible information up front).

We would predict that the Perfect Scheduler should have a higher objective score (so a higher priority score for scheduled requests), while the Dynamic Scheduler should favour shorter observations with longer availability windows, that are easier to move around.

In [1]:
import pandas as pd
import pickle
import json
import os
import datetime as dt
from matplotlib import pyplot as plt
import re
import numpy as np

In [26]:
baseline_in = "D:/lco/custom_scheduler/input_files/baseline"
baseline_out = "D:/lco/custom_scheduler/output_files/baseline"
banding_in = "D:/lco/custom_scheduler/input_files/banding"
banding_out = "D:/lco/custom_scheduler/output_files/banding"
vwt_in = "D:/lco/custom_scheduler/input_files/vwt"
vwt_out = "D:/lco/custom_scheduler/output_files/vwt"
print(os.path.isdir(baseline_in))
print(os.path.isdir(baseline_out))
print(os.path.isdir(banding_in))
print(os.path.isdir(banding_out))
print(os.path.isdir(vwt_in))
print(os.path.isdir(vwt_out))

True
True
True
True
True
True


In [27]:
filemap = {}

for dirname, folders, filenames in os.walk(baseline_in):
    for filename in filenames:
        output_filepath = os.path.join(baseline_out, filename)
        forename, ext = os.path.splitext(filename)
        perfect_filepath = os.path.join(baseline_out, forename + "_perfect" + ext)
        print(filename)
        print(os.path.isfile(output_filepath), output_filepath)
        print(os.path.isfile(perfect_filepath), perfect_filepath)
        filemap[filename + "_" + "baseline"] = {
            "input": os.path.join(dirname, filename),
            "output": output_filepath,
            "perfect": perfect_filepath,
        }

for dirname, folders, filenames in os.walk(banding_in):
    for filename in filenames:
        output_filepath = os.path.join(banding_out, filename)
        forename, ext = os.path.splitext(filename)
        perfect_filepath = os.path.join(banding_out, forename + "_perfect" + ext)
        print(filename)
        print(os.path.isfile(output_filepath), output_filepath)
        print(os.path.isfile(perfect_filepath), perfect_filepath)
        filemap[filename + "_" + "banding"] = {
            "input": os.path.join(dirname, filename),
            "output": output_filepath,
            "perfect": perfect_filepath,
        }

for dirname, folders, filenames in os.walk(vwt_in):
    for filename in filenames:
        output_filepath = os.path.join(vwt_out, filename)
        forename, ext = os.path.splitext(filename)
        perfect_filepath = os.path.join(vwt_out, forename + "_perfect" + ext)
        print(filename)
        print(os.path.isfile(output_filepath), output_filepath)
        print(os.path.isfile(perfect_filepath), perfect_filepath)
        filemap[filename + "_" + "vwt"] = {
            "input": os.path.join(dirname, filename),
            "output": output_filepath,
            "perfect": perfect_filepath,
        }

baseline_2020-08.pkl
True D:/lco/custom_scheduler/output_files/baseline\baseline_2020-08.pkl
True D:/lco/custom_scheduler/output_files/baseline\baseline_2020-08_perfect.pkl
baseline_2021-02.pkl
True D:/lco/custom_scheduler/output_files/baseline\baseline_2021-02.pkl
True D:/lco/custom_scheduler/output_files/baseline\baseline_2021-02_perfect.pkl
baseline_2021-08.pkl
True D:/lco/custom_scheduler/output_files/baseline\baseline_2021-08.pkl
True D:/lco/custom_scheduler/output_files/baseline\baseline_2021-08_perfect.pkl
baseline_2022-02.pkl
True D:/lco/custom_scheduler/output_files/baseline\baseline_2022-02.pkl
True D:/lco/custom_scheduler/output_files/baseline\baseline_2022-02_perfect.pkl
pb_2020-08_100-30-10.pkl
True D:/lco/custom_scheduler/output_files/banding\pb_2020-08_100-30-10.pkl
True D:/lco/custom_scheduler/output_files/banding\pb_2020-08_100-30-10_perfect.pkl
pb_2020-08_100-55-10.pkl
True D:/lco/custom_scheduler/output_files/banding\pb_2020-08_100-55-10.pkl
True D:/lco/custom_schedu

In [28]:
calib_proposals = [
    "OGG_calib",
    "MuSCAT Commissioning",
    "auto_focus",
    "LCOEngineering",
    "COJ_calib",
    "FLOYDS standards",
    "Photometric standards",
    "standard"
]

In [29]:
for filename in filemap:
    i = pickle.load(open(filemap[filename]["input"], "rb")) #input
    o = pickle.load(open(filemap[filename]["output"], "rb")) #output
    p = pickle.load(open(filemap[filename]["perfect"], "rb")) #perfect

    data = i["all_requests"]
    proposals = i["proposals"]
    
    data["priority"] = data.apply(lambda x: proposals[x["proposal_id"]] * x["total_duration"] * x["ipp_value"] / 60.0, axis=1)
    data["scheduled"] = data["id"].isin(o["final_completed_requests"].keys())
    data["perfect"] = data["id"].isin(p["scheduled"].keys())

    sched_both = data[data["scheduled"] & data["perfect"]]
    sched_baseline = data[data["scheduled"] & ~data["perfect"]]
    sched_perfect = data[~data["scheduled"] & data["perfect"]]
    sched_neither = data[~data["scheduled"] & ~data["perfect"]]
    
    print("FILENAME:", filename)
    print("Num Requests:", len(data))
    print("Num Baseline:", len(sched_baseline) + len(sched_both))
    print("Num Perfect:", len(sched_both) + len(sched_perfect))

    original_time = sched_both["total_duration"].sum() + sched_baseline["total_duration"].sum()
    
    requests_added = len(sched_perfect)
    mean_time_added = sched_perfect["total_duration"].mean()
    total_time_added = sched_perfect["total_duration"].sum()
    percentage_time_added = total_time_added / original_time * 100

    requests_dropped = len(sched_baseline)
    mean_time_dropped = sched_baseline["total_duration"].mean()
    total_time_dropped = sched_baseline["total_duration"].sum()
    percentage_time_dropped = total_time_dropped / original_time * 100

    total_time_difference = total_time_added - total_time_dropped
    percentage_time_difference = total_time_difference / original_time * 100

    original_priority = sched_baseline["priority"].sum() + sched_both["priority"].sum()
    added_priority = sched_perfect["priority"].sum()
    dropped_priority = sched_baseline["priority"].sum()
    net_priority_change = added_priority - dropped_priority
    percentage_priority_change = net_priority_change / original_priority * 100

    # Total Priority is Tac_Priority * IPP * Total Duration / 60.0

    print("Requests Added:", requests_added)
    print("Mean Time Added:", mean_time_added)
    print("Total Time Added:", total_time_added)
    print("% Time Added:", percentage_time_added)

    print("Requests Dropped:", requests_dropped)
    print("Mean Time Dropped:", mean_time_dropped)
    print("Total Time Dropped:", total_time_dropped)
    print("% Time Dropped:", percentage_time_dropped)

    print("Total Time Difference:", total_time_difference)
    print("% Time Difference:", percentage_time_difference)

    print("Original Priority Score:", original_priority)
    print("Added Priority:", added_priority)
    print("Dropped Priority:", dropped_priority)
    print("Net Priority Change:", net_priority_change)
    print("% Priority Change:", percentage_priority_change)

    print("===\n\n")

# 673354, 764199, 88%

# 2114996, 2868120, 74%

# 1004410, 1480906, 68%

# 1033245, 1484569, 70%

FILENAME: baseline_2020-08.pkl_baseline
Num Requests: 9254
Num Baseline: 6991
Num Perfect: 6694
Requests Added: 482
Mean Time Added: 1397.109958506224
Total Time Added: 673407
% Time Added: 6.445705260890894
Requests Dropped: 779
Mean Time Dropped: 980.7766367137356
Total Time Dropped: 764025
% Time Dropped: 7.313081037102622
Total Time Difference: -90618
% Time Difference: -0.8673757762117278
Original Priority Score: 9722180.455349844
Added Priority: 344084.3023687119
Dropped Priority: 253195.5598598337
Net Priority Change: 90888.7425088782
% Priority Change: 0.9348596534110274
===


FILENAME: baseline_2021-02.pkl_baseline
Num Requests: 16218
Num Baseline: 6157
Num Perfect: 4839
Requests Added: 914
Mean Time Added: 2314.6816192560177
Total Time Added: 2115619
% Time Added: 16.08470502439522
Requests Dropped: 2232
Mean Time Dropped: 1285.111111111111
Total Time Dropped: 2868368
% Time Dropped: 21.807732479909888
Total Time Difference: -752749
% Time Difference: -5.723027455514664
Origi

In [9]:
i["all_requests"]

,id,optimization_type,total_duration,observation_state,location,availability_windows,visibility_windows,config_data,request_group_id,ipp_value,operator,created,proposal_id,visibilities,tuples,valid_telescopes,windows,win_start,win_end
2373081,2373081,TIME,3088,WINDOW_EXPIRED,{'telescope_class': '2m0'},"[{'start': '2021-01-30T22:03:25Z', 'end': '202...","{'coj': datetime.datetime(2021, 1, 31, 10, 2, ...","[{'config_type': 'LAMP_FLAT', 'instrument_type...",1140990,1.0,SINGLE,2021-01-31 06:03:25.754436,NOAO2020B-011,"{'coj': datetime.datetime(2021, 1, 31, 10, 2, ...","((2M0-FLOYDS-SCICAM,), , , )","{'coj': ['2m0a.clma.coj'], 'ogg': ['2m0a.clma....","{'2m0a.clma.coj': datetime.datetime(2021, 1, 3...",2021-01-31 05:04:50.762038,2021-01-31 14:19:27.806779
2373019,2373019,TIME,19422,WINDOW_EXPIRED,{'telescope_class': '2m0'},"[{'start': '2021-01-31T05:05:00Z', 'end': '202...","{'coj': datetime.datetime(2021, 1, 31, 10, 2, ...","[{'config_type': 'REPEAT_EXPOSE', 'instrument_...",1140950,1.0,SINGLE,2021-01-31 03:04:57.140713,KEY2020B-005,"{'ogg': datetime.datetime(2021, 1, 31, 5, 5)(s...","((2M0-SCICAM-MUSCAT,), , , )","{'coj': ['2m0a.clma.coj'], 'ogg': ['2m0a.clma....","{'2m0a.clma.ogg': datetime.datetime(2021, 1, 3...",2021-01-31 05:05:00.000000,2021-01-31 10:30:00.000000
2373018,2373018,TIME,3388,WINDOW_EXPIRED,{'telescope_class': '2m0'},"[{'start': '2021-01-30T19:02:43Z', 'end': '202...","{'ogg': datetime.datetime(2021, 1, 31, 5, 4, 5...","[{'config_type': 'LAMP_FLAT', 'instrument_type...",1140949,1.0,SINGLE,2021-01-31 03:02:43.601319,NOAO2020B-011,"{'ogg': datetime.datetime(2021, 1, 31, 5, 4, 5...","((2M0-FLOYDS-SCICAM,), , , )","{'coj': ['2m0a.clma.coj'], 'ogg': ['2m0a.clma....","{'2m0a.clma.ogg': datetime.datetime(2021, 1, 3...",2021-01-31 05:04:50.762038,2021-01-31 08:20:29.245336
2373014,2373014,TIME,3388,WINDOW_EXPIRED,{'telescope_class': '2m0'},"[{'start': '2021-01-30T19:01:09Z', 'end': '202...","{'ogg': datetime.datetime(2021, 1, 31, 7, 13, ...","[{'config_type': 'LAMP_FLAT', 'instrument_type...",1140947,1.0,SINGLE,2021-01-31 03:01:09.805896,NOAO2020B-011,"{'ogg': datetime.datetime(2021, 1, 31, 7, 13, ...","((2M0-FLOYDS-SCICAM,), , , )","{'coj': ['2m0a.clma.coj'], 'ogg': ['2m0a.clma....","{'2m0a.clma.ogg': datetime.datetime(2021, 1, 3...",2021-01-31 07:13:29.594408,2021-01-31 16:11:55.565044
2372999,2372999,TIME,720,WINDOW_EXPIRED,"{'telescope_class': '2m0', 'site': 'coj', 'enc...","[{'start': '2021-01-31T18:02:26.887954Z', 'end...","{'coj': datetime.datetime(2021, 1, 31, 18, 2, ...","[{'config_type': 'AUTO_FOCUS', 'instrument_typ...",1140932,1.05,SINGLE,2021-01-31 02:00:28.264852,auto_focus,"{'coj': datetime.datetime(2021, 1, 31, 18, 2, ...","((2M0-SCICAM-SPECTRAL,), coj, clma, 2m0a)",{'coj': ['2m0a.clma.coj']},"{'2m0a.clma.coj': datetime.datetime(2021, 1, 3...",2021-01-31 18:02:26.887954,2021-01-31 18:32:16.281317
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2165155,2165155,TIME,417,WINDOW_EXPIRED,{'telescope_class': '2m0'},"[{'start': '2020-12-27T09:54:00Z', 'end': '202...","{'ogg': datetime.datetime(2020, 12, 28, 4, 44,...","[{'config_type': 'EXPOSE', 'instrument_type': ...",1011570,1.05,MANY,2020-06-28 21:02:10.760401,FTP2020B-003,"{'ogg': datetime.datetime(2020, 12, 28, 4, 44,...","((2M0-SCICAM-SPECTRAL,), , , )","{'coj': ['2m0a.clma.coj'], 'ogg': ['2m0a.clma....","{'2m0a.clma.ogg': datetime.datetime(2020, 12, ...",2020-12-28 04:44:52.645582,2021-01-03 08:24:50.411309
2165156,2165156,TIME,417,WINDOW_EXPIRED,{'telescope_class': '2m0'},"[{'start': '2021-01-03T09:54:00Z', 'end': '202...","{'ogg': datetime.datetime(2021, 1, 4, 4, 48, 5...","[{'config_type': 'EXPOSE', 'instrument_type': ...",1011570,1.05,MANY,2020-06-28 21:02:10.760401,FTP2020B-003,"{'ogg': datetime.datetime(2021, 1, 4, 4, 48, 5...","((2M0-SCICAM-SPECTRAL,), , , )","{'coj': ['2m0a.clma.coj'], 'ogg': ['2m0a.clma....","{'2m0a.clma.ogg': datetime.datetime(2021, 1, 4...",2021-01-04 04:48:56.983308,2021-01-10 07:57:19.508462
21